# IncidentCommander — RL Training (Colab T4 / A100)

**Stack:** Unsloth · Qwen2.5-1.5B-Instruct (4-bit QLoRA actor) · Qwen2.5-72B-Instruct critic via Hugging Face Inference Providers · PPO + GAE

This notebook trains the IncidentCommander SRE agent against the in-repo simulator (**381 deterministic scenarios** including 166 saboteur/Slack scenarios, cascading topology, K8s adversary, runbook traps).

## Why this critic
Claude Haiku 4.5 is **not** hosted on Hugging Face — Anthropic and HF are different vendors. Instead we use **Qwen2.5-72B-Instruct via the HF Inference Providers router**, which (a) is free with any HF account, (b) is ~48× the size of the actor, giving the value head genuine compute headroom, and (c) is routinely served through Together / Nebius for free credits. The same code path also accepts `meta-llama/Meta-Llama-3.1-70B-Instruct` or `mistralai/Mistral-Large-2407`.

## How to run
1. **Runtime → Change runtime type → T4 GPU (or A100)**
2. Run the cells top-to-bottom.
3. Set your `HF_TOKEN` when prompted (free tier works).

## 1 · Verify GPU + install dependencies

In [ ]:
!nvidia-smi || echo 'No GPU — switch the runtime to T4 or A100.'

In [ ]:
%pip install -q --upgrade pip
%pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
%pip install -q --no-deps 'xformers<0.0.27' trl peft accelerate bitsandbytes
%pip install -q 'huggingface_hub>=0.25' pydantic==2.* httpx

## 2 · Get the IncidentCommander repo into Colab

**Option A (recommended):** push the repo to GitHub (`scripts/push_to_remotes.ps1` does that for you) and set `IC_REPO_URL` below.

**Option B:** zip the local repo (`Compress-Archive -Path .\rl-agent, .\colab -DestinationPath ic.zip`) and drag it into Colab's `/content/` folder as `incident-commander.zip`.

**Option C:** clone from a Hugging Face Space repo via `IC_HF_SPACE`.

In [ ]:
import os, subprocess, sys, zipfile
REPO_DIR = '/content/incident-commander'

IC_REPO_URL = os.environ.get('IC_REPO_URL', '')
if IC_REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--depth','1', IC_REPO_URL, REPO_DIR], check=True)

if not os.path.isdir(REPO_DIR) and os.path.exists('/content/incident-commander.zip'):
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile('/content/incident-commander.zip') as z:
        z.extractall(REPO_DIR)

IC_HF_SPACE = os.environ.get('IC_HF_SPACE', '')
if not os.path.isdir(REPO_DIR) and IC_HF_SPACE:
    subprocess.run(['git','clone',
                    f'https://huggingface.co/spaces/{IC_HF_SPACE}', REPO_DIR], check=True)

assert os.path.isdir(REPO_DIR), 'No repo present — pick Option A/B/C above.'
%cd /content/incident-commander
sys.path.insert(0, '/content/incident-commander')
sys.path.insert(0, '/content/incident-commander/rl-agent')

## 3 · Hugging Face token

`HF_TOKEN` powers (a) actor weight downloads and (b) the Qwen2.5-72B critic over the Inference Providers router. Get one at https://huggingface.co/settings/tokens — **Read** scope is enough for training (add **Write** if you want this notebook to push your trained adapter).

> **Security:** rotate the two HF tokens that were leaked in earlier chat (`hf_RLF…`, `hf_IBf…`).

In [ ]:
import os, getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Paste your HF token (hf_…): ')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
os.environ['INCIDENT_COMMANDER_MOCK'] = 'true'
from huggingface_hub import login
login(os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF login OK')

## 4 · Smoke-test the simulator

In [ ]:
import sys, glob, collections
sys.path.insert(0, '/content/incident-commander/rl-agent')
from simulator import SimState, dispatch, load_scenario
files = sorted(glob.glob('rl-agent/scenarios/sim/*/*.json'))
by_diff = collections.Counter(p.split('/')[-2] for p in files)
ok = 0
for p in files:
    s = SimState(); scn = load_scenario(p, s)
    last = None
    for step in scn['correct_action_chain']:
        last = dispatch(step, s)
    if last and last.ok: ok += 1
print(f'{ok}/{len(files)} scenario chains pass.  By difficulty: {dict(by_diff)}')

## 5 · Sanity-check the HF Inference critic

In [ ]:
from colab.train_lib import LLMCritic
critic = LLMCritic(provider='hf', model='Qwen/Qwen2.5-72B-Instruct')
v_good = critic.value(
    observation='{"task":"users_db is dead, agent has not failed over yet"}',
    action={'id': 'platform.failover_replica', 'params': {'target': 'users_db'}})
v_bad  = critic.value(
    observation='{"task":"users_db is unhealthy, runbook NOT read"}',
    action={'id': 'platform.restart_cluster', 'params': {'target': 'users_db'}})
print(f'failover V={v_good:.2f}    restart-brute V={v_bad:.2f}')
assert v_good >= v_bad, 'Critic should rate failover above brute-force restart.'

## 6 · Configure + run training

In [ ]:
from colab.train_lib import CFG, train_loop
CFG.update({
    'total_updates':       40,    # bump to 80–120 for the real run
    'rollouts_per_update': 4,
    'max_steps_per_ep':    14,
    'critic_provider':     'hf',
    'critic_model':        'Qwen/Qwen2.5-72B-Instruct',
    'lr':                  1e-5,
    'run_name':            'demo01',
    'tasks': [
        'sim_easy_lambda_throttle_001',
        'sim_med_eb_lambda_016',
        'sim_hard_apigw_chain_001',
        'sim_advanced_cascade_users_db_001',
        'sim_advanced_runbook_trap_postgres_001',
        'sim_advanced_trolley_orders_db_001',
        'sim_advanced_saboteur_duel_001',
        'sim_advanced_slack_redherring_001',
        'sim_gen_app_leak_checkout_005',
        'sim_gen_db_duel_users_db_003',
        'sim_gen_redherring_payments_007',
        'sim_gen_cascade_payments_db_004',
    ],
})
log_path = train_loop()
print('Training log:', log_path)

## 7 · Quick visualization of training curves

In [ ]:
import json, matplotlib.pyplot as plt
data = json.load(open(log_path))
u = [e['update']      for e in data['updates']]
r = [e['mean_reward'] for e in data['updates']]
v = [e['mean_value']  for e in data['updates']]
k = [e['ppo']['kl']   for e in data['updates']]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(u, r, color='#3fb950'); ax[0].set_title('mean_reward'); ax[0].grid(alpha=.3)
ax[1].plot(u, v, color='#58a6ff'); ax[1].set_title('mean_value (Qwen-72B critic)'); ax[1].grid(alpha=.3)
ax[2].plot(u, k, color='#f85149'); ax[2].set_title('PPO KL'); ax[2].grid(alpha=.3)
for a in ax: a.set_xlabel('update')
plt.tight_layout(); plt.show()

## 8 · Replay artifact with the trained agent

In [ ]:
import glob
from colab.train_lib import IncidentRolloutCollector, QwenActor, LLMCritic, CFG
actor = QwenActor(model_name=CFG['actor_model'], max_seq_len=CFG['max_seq_len'],
                  lora_r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
                  lora_dropout=CFG['lora_dropout'])
ckpts = sorted(glob.glob('colab/logs/adapter_*_final')) or sorted(glob.glob('colab/logs/adapter_*'))
if ckpts:
    actor.model.load_adapter(ckpts[-1], adapter_name='trained')
    actor.model.set_adapter('trained')
    print('Loaded', ckpts[-1])
critic = LLMCritic(provider=CFG['critic_provider'], model=CFG['critic_model'])
collector = IncidentRolloutCollector(actor, critic,
                                      tasks=['sim_advanced_saboteur_duel_001'],
                                      max_steps_per_ep=14)
_ = collector.collect(1)
replays = sorted(glob.glob('rl-agent/replays/*.html'))
print('Latest replay:', replays[-1] if replays else 'none')

## 9 · Push trained adapter + logs back to Hugging Face

Set `IC_HF_USER` to your HF username before running. Creates (or re-uses) a public model repo `<user>/incident-commander-actor`.

In [ ]:
import os, glob
from huggingface_hub import HfApi, create_repo
IC_HF_USER = os.environ.get('IC_HF_USER', '')
if not IC_HF_USER:
    print('Skipping push — set os.environ["IC_HF_USER"] to enable.')
else:
    api  = HfApi()
    repo = f'{IC_HF_USER}/incident-commander-actor'
    create_repo(repo, exist_ok=True, repo_type='model')
    final = sorted(glob.glob('colab/logs/adapter_*_final'))[-1]
    api.upload_folder(folder_path=final, repo_id=repo, repo_type='model',
                      path_in_repo='adapter')
    api.upload_folder(folder_path='colab/logs', repo_id=repo, repo_type='model',
                      path_in_repo='logs', allow_patterns=['*.json'])
    api.upload_folder(folder_path='rl-agent/replays', repo_id=repo,
                      repo_type='model', path_in_repo='replays',
                      allow_patterns=['*.html'])
    print(f'Pushed → https://huggingface.co/{repo}')

## 10 · Download artifacts to your laptop

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/ic_artifacts', 'zip',
                    root_dir='/content/incident-commander', base_dir='colab/logs')
shutil.make_archive('/content/ic_replays',  'zip',
                    root_dir='/content/incident-commander', base_dir='rl-agent/replays')
files.download('/content/ic_artifacts.zip')
files.download('/content/ic_replays.zip')